# Evaluation of AdvanceRAG using Ragas

In [27]:
import os
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
import json
import torch
import re
from pathlib import Path
from datasets import Dataset

In [28]:
# Load retrieved dataset
input_path = Path("../datas/evaluation_dataset_retrieved_graphrag.jsonl")
dataset_items = []

with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dataset_items.append(json.loads(line))

print(f"Loaded {len(dataset_items)} evaluation records.")

Loaded 10 evaluation records.


In [29]:
# def clean_answer(text):
#     if not text:
#         return ""
#     # Remove <think>...</think> blocks
#     return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

# Prepare data for Ragas
# Mapping to both Ragas v0.1 and v0.2 naming standards to ensure compatibility
data = {
    "user_input": [item["question"] for item in dataset_items],
    # "answer": [clean_answer(item["rag_answer"]) for item in dataset_items],
    "contexts": [item["retrieved_contexts"] for item in dataset_items],
    "ground_truth": [item["ground_truth"] for item in dataset_items],
}

dataset = Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['user_input', 'contexts', 'ground_truth'],
    num_rows: 10
})


In [30]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings


eval_llm = ChatOpenAI(
    model="qwen/qwen3-1.7b",
    base_url="http://localhost:1234/v1",
    api_key="abc",
    temperature=0.0,
    # extra_body={
    #     "chat_template_kwargs": {
    #         "enable_thinking": False
    #     }
    # },
)

# Configure Embeddings for Ragas evaluation
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_embeddings = HuggingFaceEmbeddings(
    model_name="minhthuan77f1/binhdinh-embedding",
    model_kwargs={"device": device}
)

In [31]:
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    # faithfulness,
    # answer_relevancy,
    context_precision,
    context_recall,
)

# Define metrics
metrics = [
    # faithfulness,
    # answer_relevancy,
    context_precision,
    context_recall,
]

# Run evaluation using local LLM and Embeddings
print("Evaluating dataset...")
results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings,
    run_config=RunConfig(max_workers=2, timeout=240, max_retries=2)
)

C:\Users\Windows\AppData\Local\Temp\ipykernel_19712\2556198911.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\Windows\AppData\Local\Temp\ipykernel_19712\2556198911.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (


Evaluating dataset...


Evaluating: 100%|██████████| 20/20 [04:47<00:00, 14.39s/it]


In [32]:
# Print average results
print(results)

{'context_precision': 0.7000, 'context_recall': 0.8000}


In [33]:
# Save and display detailed results
output_dir = Path("../results")
output_dir.mkdir(exist_ok=True)

# Convert results to a pandas DataFrame
df_results = results.to_pandas()

# Display the detailed dataframe in the notebook
display(df_results)

# Save to CSV
df_results.to_csv(output_dir / "ragas_eval_details_graphrag.csv", index=False)

# Save summary metrics safely by calculating the mean of metric columns from the dataframe
metric_names = [m.name for m in metrics]
summary = {}
for col in df_results.columns:
    if col in metric_names:
        summary[col] = float(df_results[col].mean())

with open(output_dir / "ragas_eval_summary_graphrag.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)

print(f"Details saved to {output_dir / 'ragas_eval_details_graphrag.csv'}")
print(f"Summary saved to {output_dir / 'ragas_eval_summary_graphrag.json'}")

,user_input,retrieved_contexts,reference,context_precision,context_recall
0,Nhồi máu cơ tim cấp có thể gây ra biến chứng n...,[Nhịp tim nhanh do nguyên nhân sinh lý hoặc bệ...,Biến chứng liên quan đến nhịp tim là rối loạn ...,1.0,1.0
1,Cặp vợ chồng nào cần đi khám vô sinh hiếm muộn...,[Khám hiếm muộn thường được khuyến cáo với nhữ...,"Theo tài liệu, cặp vợ chồng nên đi khám vô sin...",1.0,1.0
2,Viêm võng mạc sắc tố có thể gây ra những triệu...,[được kiểm soát gây tắc nghẽn mạch máu nuôi dư...,Triệu chứng của viêm võng mạc sắc tố thường bắ...,0.0,0.0
3,Bệnh nhân rung nhĩ cần làm gì nếu muốn đảm bảo...,[tĩnh mạch hoặc đường uống để thiết lập lại nh...,Người bệnh cần tuân thủ sử dụng thuốc đúng the...,1.0,1.0
4,Người bình thường có thể bị hạ đường huyết sau...,[Hạ đường huyết bất thường sau ăn do tăng tiết...,"Có, người bình thường cũng có thể bị hạ đường ...",1.0,1.0
5,Chi phí giảm béo toàn thân tại Trung tâm Kiểm ...,[Chi phí giảm béo toàn thân khoảng 20 – 25 tri...,Chi phí giảm béo toàn thân tại Trung tâm Kiểm ...,1.0,1.0
6,Bệnh nhân Nguyễn Quốc Linh bị suy thận trái do...,[Tương tự như các phương pháp xét nghiệm chẩn ...,Vì thận trái của bệnh nhân Linh không thể đảm ...,0.0,0.0
7,Sởi có phải là bệnh da liễu không?,[],Sởi không phải là bệnh da liễu. Sởi là bệnh nh...,0.0,1.0
8,Ung thư vú luminal B có phổ biến hay không?,[Nhiều yếu tố góp phần vào tiên lượng của ung ...,Ung thư vú luminal B chiếm khoảng 15%-20% tron...,1.0,1.0
9,Bà bầu ra khí hư màu trắng sữa không ngứa có b...,"[Trong thai kỳ, khí hư màu trắng sữa thường là...","Trong thai kỳ, khí hư màu trắng sữa là hiện tư...",1.0,1.0


Details saved to ..\results\ragas_eval_details_graphrag.csv
Summary saved to ..\results\ragas_eval_summary_graphrag.json
